# Kapitel 19.5 - WSGI Praxis, Architektur, Deployment und Uebungen

Dieses Abschlussnotebook verbindet Technik, Architektur und Projektpraxis.
Du lernst, wie aus einem Lernbeispiel ein sauber strukturiertes Webprojekt wird.

# Lernziele

Nach diesem Notebook kannst du:

- WSGI-Projekte in Schichten strukturieren
- einfache Middleware-Ideen umsetzen
- JSON-Antworten korrekt liefern
- Konfiguration und Deployment-Grundlagen erklaeren
- groessere Uebungen mit professioneller Denke loesen

# Voraussetzungen

Du solltest mitbringen:

- Verstaendnis fuer Routing und Request-Verarbeitung
- sichere Nutzung von Funktionen und Modulen
- Grundwissen zu Fehlerbehandlung

Empfehlung: Notebook 19.4 vorher komplett durcharbeiten.

# Theorie

## Schichtenmodell (einfach)

1. Transportebene: HTTP/WSGI
2. Routingebene: Pfad -> Handler
3. Fachlogik: eigentliche Regeln
4. Praesentationslogik: Text/HTML/JSON

## Middleware-Idee

Middleware liegt zwischen Server und App, z. B. fuer Logging, Auth oder Header-Anpassung.

## Deployment-Grundlagen

In der Praxis laufen WSGI-Apps oft mit Gunicorn/uWSGI hinter einem Reverse Proxy wie Nginx.
Fuer den Kurs reicht das konzeptionelle Verstaendnis.

# Erklaerung

Ein gutes Lernprojekt sollte frueh folgende Fragen beantworten:

- Wo liegt Routing?
- Wo wird validiert?
- Wo wird formatiert (HTML/JSON)?
- Wie werden Fehler zentral behandelt?

Diese Klarheit reduziert langfristig technische Schulden.

# Syntax

## JSON-Antwort in WSGI

```python
import json
data = {'status': 'ok'}
body = json.dumps(data).encode('utf-8')
```

## Middleware-Muster

```python
def middleware(app):
    def wrapped(environ, start_response):
        return app(environ, start_response)
    return wrapped
```

# Merke

- Gute Architektur startet mit klaren Verantwortlichkeiten.
- Zentralisierte Fehlerbehandlung spart spaeter viel Zeit.
- Logging und Monitoring sind keine Extras, sondern Betriebsgrundlagen.

# Parameter

Bei Middleware und Response-Buildern sind haeufig:

- `environ` (Request-Kontext)
- `start_response` (Status und Header)
- `data` (fachliche Nutzlast)
- `status` (z. B. `200 OK`, `400 Bad Request`)

entscheidend.

# Rueckgabewert

Auch bei komplexeren Mustern gilt:
- Die finale WSGI-App liefert ein bytes-Iterable zurueck.
- Alle Zwischenschritte (Routing, Middleware, Formatter) arbeiten darauf hin.

In [ ]:
# Beispiel 1: JSON-Response-Builder
import json

def json_response(start_response, data, status='200 OK'):
    body = json.dumps(data, ensure_ascii=False).encode('utf-8')
    headers = [('Content-Type', 'application/json; charset=utf-8')]
    start_response(status, headers)
    return [body]

def fake_start_response(status, headers):
    print('STATUS :', status)
    print('HEADERS:', headers)

result = json_response(fake_start_response, {'kurs': 'WSGI', 'level': 'fortgeschritten'})
print(result[0].decode('utf-8'))

# Beispiel 1 - Erklaerung

Ein zentraler Response-Builder verhindert Duplikate und Inkonsistenzen.
Besonders bei APIs ist das ein grosser Qualitaetsgewinn.

In [ ]:
# Beispiel 2: Logging-Middleware
from datetime import datetime

def logging_middleware(app):
    def wrapped(environ, start_response):
        method = environ.get('REQUEST_METHOD', 'GET')
        path = environ.get('PATH_INFO', '/')
        print(f"[LOG {datetime.now().strftime('%H:%M:%S')}] {method} {path}")
        return app(environ, start_response)
    return wrapped

def app_simple(environ, start_response):
    return json_response(start_response, {'nachricht': 'Middleware aktiv'})

app_mit_logging = logging_middleware(app_simple)
body = app_mit_logging({'REQUEST_METHOD': 'GET', 'PATH_INFO': '/api/info'}, fake_start_response)
print(body[0].decode('utf-8'))

# Beispiel 2 - Erklaerung

Die Middleware kapselt Querschnittslogik, ohne Handler-Code zu verschmutzen.
Das ist ein zentraler Architekturvorteil in produktiven Systemen.

In [ ]:
# Beispiel 3: Kleine API-App mit Routing + Fehlerantwort
def api_app(environ, start_response):
    path = environ.get('PATH_INFO', '/')

    if path == '/api/ping':
        return json_response(start_response, {'pong': True})

    if path == '/api/version':
        return json_response(start_response, {'version': '1.0.0', 'name': 'kurs-api'})

    return json_response(start_response, {'fehler': 'Nicht gefunden'}, status='404 Not Found')

api_mit_logging = logging_middleware(api_app)
print(api_mit_logging({'REQUEST_METHOD': 'GET', 'PATH_INFO': '/api/ping'}, fake_start_response)[0].decode('utf-8'))
print(api_mit_logging({'REQUEST_METHOD': 'GET', 'PATH_INFO': '/api/unknown'}, fake_start_response)[0].decode('utf-8'))

# Beispiel 3 - Erklaerung

Hier siehst du eine API-Denkweise mit klaren Endpunkten und JSON-Ausgaben.
Auch in kleinen Projekten lohnt sich diese Struktur fuer Wartbarkeit und Tests.

# Praxisbeispiel

Projektidee: Kursverwaltung als Mini-API

- `/api/kurse` liefert Kursliste
- `/api/kurs?id=...` liefert ein einzelnes Objekt
- Fehlerfall bei unbekannter ID

In [ ]:
from urllib.parse import parse_qs

KURSE = [
    {'id': 1, 'titel': 'Python Grundlagen'},
    {'id': 2, 'titel': 'Web mit WSGI'},
    {'id': 3, 'titel': 'Datenanalyse Einstieg'}
]

def kurs_api(environ, start_response):
    path = environ.get('PATH_INFO', '/')

    if path == '/api/kurse':
        return json_response(start_response, {'kurse': KURSE})

    if path == '/api/kurs':
        query = parse_qs(environ.get('QUERY_STRING', ''))
        kurs_id_text = query.get('id', [''])[0]

        if not kurs_id_text.isdigit():
            return json_response(start_response, {'fehler': 'id muss numerisch sein'}, status='400 Bad Request')

        kurs_id = int(kurs_id_text)
        for kurs in KURSE:
            if kurs['id'] == kurs_id:
                return json_response(start_response, {'kurs': kurs})

        return json_response(start_response, {'fehler': 'Kurs nicht gefunden'}, status='404 Not Found')

    return json_response(start_response, {'fehler': 'Route unbekannt'}, status='404 Not Found')

print(kurs_api({'PATH_INFO': '/api/kurse'}, fake_start_response)[0].decode('utf-8'))
print(kurs_api({'PATH_INFO': '/api/kurs', 'QUERY_STRING': 'id=2'}, fake_start_response)[0].decode('utf-8'))
print(kurs_api({'PATH_INFO': '/api/kurs', 'QUERY_STRING': 'id=99'}, fake_start_response)[0].decode('utf-8'))

# Haeufige Fehler

1. Unterschied zwischen 400 und 404 nicht beachten.
2. Route- und Datenlogik unnoetig vermischen.
3. Keine zentrale Response-Funktion verwenden.
4. Fehlende Logging-Strategie bei Produktionseinfuehrung.
5. Konfiguration direkt hart codieren statt ueber Konstanten/Umgebung.

# Best Practice

- Trenne klar zwischen HTTP-Schicht und Fachlogik.
- Nutze Middleware fuer Logging, Security-Header und Messwerte.
- Definiere konsistente API-Fehlerobjekte.
- Dokumentiere Endpunkte inklusive Beispielantworten.
- Plane frueh, wie Konfiguration pro Umgebung (Dev/Test/Prod) funktioniert.

# Tipp

Wenn du spaeter auf Flask wechselst, versuche die Schichten beizubehalten.
Dann bleibt dein Projekt auch bei wachsender Komplexitaet gut beherrschbar.

# Uebung

Implementiere eine kleine Aufgaben-API mit folgenden Routen:

1. `/api/aufgaben` -> Liste aller Aufgaben
2. `/api/aufgabe?id=...` -> einzelne Aufgabe
3. Bei ungueltiger ID -> `400 Bad Request`
4. Bei unbekannter ID -> `404 Not Found`
5. Middleware-Logging fuer jeden Request

In [ ]:
# Loesung
AUFGABEN = [
    {'id': 1, 'titel': 'CGI Header verstehen'},
    {'id': 2, 'titel': 'WSGI Routing bauen'},
    {'id': 3, 'titel': 'Input validieren'}
]

def aufgaben_api(environ, start_response):
    path = environ.get('PATH_INFO', '/')

    if path == '/api/aufgaben':
        return json_response(start_response, {'aufgaben': AUFGABEN})

    if path == '/api/aufgabe':
        query = parse_qs(environ.get('QUERY_STRING', ''))
        id_text = query.get('id', [''])[0]

        if not id_text.isdigit():
            return json_response(start_response, {'fehler': 'id muss numerisch sein'}, status='400 Bad Request')

        gesuchte_id = int(id_text)
        for eintrag in AUFGABEN:
            if eintrag['id'] == gesuchte_id:
                return json_response(start_response, {'aufgabe': eintrag})

        return json_response(start_response, {'fehler': 'Aufgabe nicht gefunden'}, status='404 Not Found')

    return json_response(start_response, {'fehler': 'Route unbekannt'}, status='404 Not Found')

aufgaben_api_logged = logging_middleware(aufgaben_api)
print(aufgaben_api_logged({'REQUEST_METHOD': 'GET', 'PATH_INFO': '/api/aufgaben'}, fake_start_response)[0].decode('utf-8'))
print(aufgaben_api_logged({'REQUEST_METHOD': 'GET', 'PATH_INFO': '/api/aufgabe', 'QUERY_STRING': 'id=2'}, fake_start_response)[0].decode('utf-8'))
print(aufgaben_api_logged({'REQUEST_METHOD': 'GET', 'PATH_INFO': '/api/aufgabe', 'QUERY_STRING': 'id=abc'}, fake_start_response)[0].decode('utf-8'))

# Zusammenfassung

Kapitel 19 ist damit inhaltlich komplett aufgebaut:

- CGI von Grundlagen bis sichere Praxis
- WSGI von Signatur bis Mini-Framework
- Architektur- und Deployment-Denken fuer realistische Projekte
- viele Uebungen mit nachvollziehbaren Loesungen

Damit hast du eine starke Basis fuer den naechsten Schritt: moderne Frameworks und performantere Serverarchitekturen.

# Weiterfuehrende Links

- Gunicorn Dokumentation
- uWSGI Dokumentation
- Nginx als Reverse Proxy (Grundlagen)
- Flask Deployment Guide

## Technischer Tiefgang

Der Fokus liegt auf reproduzierbaren technischen Entscheidungen statt auf isolierten Einzelbeispielen.
Dabei werden Architektur, Robustheit und Betriebsfaehigkeit gemeinsam betrachtet.

## Zentrale Fachbegriffe

HTTP Semantics
Status Code Family
WSGI Callable
Request Lifecycle
Input Sanitization
Header Validation

In [ ]:
# WSGI-Minibeispiel mit Statuscode
def app(environ, start_response):
    path = environ.get("PATH_INFO", "/")
    if path == "/health":
        start_response("200 OK", [("Content-Type", "text/plain")])
        return [b"ok"]
    start_response("404 Not Found", [("Content-Type", "text/plain")])
    return [b"not found"]

## Fallstudie (Praxis)

Waehle ein realistisches Produktionsszenario und beschreibe systematisch Ursache, Risiko und technische Gegenmassnahmen.
Ergaenze mindestens ein Kriterium fuer Monitoring und ein Kriterium fuer Release-Entscheidungen.

## Haeufige Fehler und Debugging-Checkliste

- Ist das Problem reproduzierbar mit klaren Schritten?
- Sind relevante Signale vorhanden (Logs, Tests, Metriken)?
- Wurde eine konkrete Hypothese getestet und falsifiziert/bestaetigt?
- Ist die Korrektur durch einen Regressionstest abgesichert?
- Wurden Betriebsfolgen und Dokumentation mit aktualisiert?

## Pruefungsfragen und Kurzloesungen

1. Warum ist Reproduzierbarkeit in Fehleranalyse und Betrieb zentral?
Kurzloesung: Ohne reproduzierbare Befunde sind Ursachenanalyse, Fix und Absicherung nicht belastbar.
2. Was unterscheidet technische Begriffe von bloessem Buzzword-Einsatz?
Kurzloesung: Praezise Begriffe steuern messbare Entscheidungen und verbessern Teamkommunikation.
3. Welche Mindestkriterien sollte ein Release-Gate enthalten?
Kurzloesung: Teststatus, Sicherheitschecks, Fehlerbudget und nachvollziehbare Freigabeentscheidung.